# 03 — Forecasting Model Training & Evaluation
**DRM Key Demand Forecasting**

This notebook trains and evaluates two complementary forecasting approaches, then selects the best model for production use.

---

**Models:**
1. **Facebook Prophet** — Handles seasonality, holidays, and trend changes with minimal tuning
2. **LightGBM** — Gradient-boosted trees using engineered lag/calendar features

**Evaluation:**
- Train/test split: last 30 days held out
- Metrics: MAPE, RMSE, MAE
- Visual comparison of forecasts vs actuals

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from prophet import Prophet
import lightgbm as lgb
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)
import warnings
import os
import pickle

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})

FORECAST_HORIZON = 30
RANDOM_STATE = 42

print('Libraries loaded. Forecast horizon:', FORECAST_HORIZON, 'days')

In [ ]:
DATA_DIR = os.path.join('..', 'Dataset')

df = pd.read_csv(
    os.path.join(DATA_DIR, 'daily_drm_keys_clean.csv'),
    parse_dates=['LogDate']
)

print(f'Loaded: {len(df)} days | {df["LogDate"].min().date()} → {df["LogDate"].max().date()}')

## 1. Train / Test Split

In [ ]:
split_date = df['LogDate'].max() - pd.Timedelta(days=FORECAST_HORIZON)

df_train = df[df['LogDate'] <= split_date].copy()
df_test  = df[df['LogDate'] > split_date].copy()

print(f'Train: {len(df_train)} days | {df_train["LogDate"].min().date()} → {df_train["LogDate"].max().date()}')
print(f'Test : {len(df_test)} days  | {df_test["LogDate"].min().date()} → {df_test["LogDate"].max().date()}')

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_train['LogDate'], df_train['Total_Daily_Keys'], label='Train', color='steelblue')
ax.plot(df_test['LogDate'], df_test['Total_Daily_Keys'], label='Test', color='crimson')
ax.axvline(split_date, color='black', linestyle='--', alpha=0.5, label='Split')
ax.set_title(f'Train/Test Split (last {FORECAST_HORIZON} days held out)')
ax.set_ylabel('DRM Keys')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Model A — Facebook Prophet

Prophet excels at:
- Automatic weekly and yearly seasonality
- Handling trend changepoints
- Built-in uncertainty intervals

In [ ]:
# Prophet requires columns named 'ds' (date) and 'y' (target)
prophet_train = df_train[['LogDate', 'Total_Daily_Keys']].rename(
    columns={'LogDate': 'ds', 'Total_Daily_Keys': 'y'}
)

model_prophet = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    interval_width=0.90
)

model_prophet.fit(prophet_train)
print('Prophet model fitted.')

In [ ]:
# Generate forecast for test period
future = model_prophet.make_future_dataframe(periods=FORECAST_HORIZON, freq='D')
forecast_prophet = model_prophet.predict(future)

# Extract test-period predictions
pred_prophet = forecast_prophet[forecast_prophet['ds'] > split_date][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
pred_prophet = pred_prophet.merge(
    df_test[['LogDate', 'Total_Daily_Keys']].rename(columns={'LogDate': 'ds', 'Total_Daily_Keys': 'actual'}),
    on='ds'
)

display(pred_prophet.head())

In [ ]:
# Prophet components plot
fig = model_prophet.plot_components(forecast_prophet)
plt.suptitle('Prophet — Decomposed Components', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Prophet forecast vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pred_prophet['ds'], pred_prophet['actual'], 'o-', label='Actual', color='steelblue', markersize=4)
ax.plot(pred_prophet['ds'], pred_prophet['yhat'], 's-', label='Prophet Forecast', color='darkorange', markersize=4)
ax.fill_between(
    pred_prophet['ds'], pred_prophet['yhat_lower'], pred_prophet['yhat_upper'],
    alpha=0.2, color='darkorange', label='90% CI'
)
ax.set_title(f'Prophet — {FORECAST_HORIZON}-Day Forecast vs Actual')
ax.set_ylabel('DRM Keys')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.show()

---
## 3. Model B — LightGBM

Tree-based model using hand-crafted time-series features. Captures non-linear relationships and complex interactions between calendar/lag features.

In [ ]:
FEATURES = [
    'DayOfWeek', 'IsWeekend', 'DayOfMonth', 'WeekOfYear',
    'Month', 'Quarter', 'IsMonthStart', 'IsMonthEnd',
    'Lag_7', 'Lag_14', 'Lag_30',
    'MA_7', 'MA_14', 'MA_30', 'Std_7'
]
TARGET = 'Total_Daily_Keys'

# Drop rows where lag/rolling features are NaN (initial warmup period)
df_model = df.dropna(subset=FEATURES).copy()

train_mask = df_model['LogDate'] <= split_date
X_train = df_model.loc[train_mask, FEATURES]
y_train = df_model.loc[train_mask, TARGET]
X_test  = df_model.loc[~train_mask, FEATURES]
y_test  = df_model.loc[~train_mask, TARGET]

print(f'LightGBM — Train: {len(X_train)} | Test: {len(X_test)} | Features: {len(FEATURES)}')

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': RANDOM_STATE,
    'verbose': -1
}

train_set = lgb.Dataset(X_train, y_train)
val_set   = lgb.Dataset(X_test, y_test, reference=train_set)

model_lgb = lgb.train(
    params,
    train_set,
    num_boost_round=1000,
    valid_sets=[train_set, val_set],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print(f'Best iteration: {model_lgb.best_iteration}')

In [ ]:
pred_lgb = model_lgb.predict(X_test)

# LightGBM forecast vs actual
test_dates = df_model.loc[~train_mask, 'LogDate'].values

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates, y_test.values, 'o-', label='Actual', color='steelblue', markersize=4)
ax.plot(test_dates, pred_lgb, 's-', label='LightGBM Forecast', color='green', markersize=4)
ax.set_title(f'LightGBM — {FORECAST_HORIZON}-Day Forecast vs Actual')
ax.set_ylabel('DRM Keys')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': model_lgb.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance['Feature'], importance['Importance'], color='steelblue')
ax.set_title('LightGBM — Feature Importance (gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 4. Model Comparison & Evaluation

In [ ]:
def evaluate_model(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = mean_absolute_percentage_error(actual, predicted) * 100
    return {
        'Model': model_name,
        'MAE': round(mae, 1),
        'RMSE': round(rmse, 1),
        'MAPE (%)': round(mape, 2)
    }

# Align predictions to same test set
actual_prophet = pred_prophet['actual'].values
predicted_prophet = pred_prophet['yhat'].values

actual_lgb = y_test.values
predicted_lgb_vals = pred_lgb

results = pd.DataFrame([
    evaluate_model(actual_prophet, predicted_prophet, 'Prophet'),
    evaluate_model(actual_lgb, predicted_lgb_vals, 'LightGBM')
])

print('\n' + '=' * 55)
print('MODEL EVALUATION — Test Set Performance')
print('=' * 55)
display(results)

best_model_name = results.loc[results['MAPE (%)'].idxmin(), 'Model']
print(f'\n→ Best model by MAPE: {best_model_name}')

In [ ]:
# Side-by-side comparison
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(pred_prophet['ds'], pred_prophet['actual'], 'o-', label='Actual', color='steelblue', markersize=5, linewidth=2)
ax.plot(pred_prophet['ds'], pred_prophet['yhat'], 's--', label='Prophet', color='darkorange', markersize=4)
ax.plot(test_dates, pred_lgb, '^--', label='LightGBM', color='green', markersize=4)
ax.fill_between(
    pred_prophet['ds'], pred_prophet['yhat_lower'], pred_prophet['yhat_upper'],
    alpha=0.15, color='darkorange', label='Prophet 90% CI'
)

ax.set_title('Model Comparison — Prophet vs LightGBM')
ax.set_ylabel('DRM Keys')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

resid_prophet = actual_prophet - predicted_prophet
resid_lgb = actual_lgb - predicted_lgb_vals

axes[0].bar(range(len(resid_prophet)), resid_prophet, color='darkorange', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Prophet — Residuals')
axes[0].set_xlabel('Test Day')
axes[0].set_ylabel('Actual - Predicted')

axes[1].bar(range(len(resid_lgb)), resid_lgb, color='green', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('LightGBM — Residuals')
axes[1].set_xlabel('Test Day')
axes[1].set_ylabel('Actual - Predicted')

plt.tight_layout()
plt.show()

## 5. Save Best Model & Forecast for Next Period

In [ ]:
# Retrain Prophet on ALL data for production forecast
prophet_full = df[['LogDate', 'Total_Daily_Keys']].rename(
    columns={'LogDate': 'ds', 'Total_Daily_Keys': 'y'}
)

model_prophet_final = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    interval_width=0.90
)
model_prophet_final.fit(prophet_full)

# Forecast next 30 days
future_30 = model_prophet_final.make_future_dataframe(periods=30, freq='D')
forecast_30 = model_prophet_final.predict(future_30)

# Extract only future dates
last_date = df['LogDate'].max()
forecast_future = forecast_30[forecast_30['ds'] > last_date][[
    'ds', 'yhat', 'yhat_lower', 'yhat_upper'
]].copy()
forecast_future.columns = ['Date', 'Forecast', 'Lower_90', 'Upper_90']
forecast_future['Forecast'] = forecast_future['Forecast'].round(0).astype(int)
forecast_future['Lower_90'] = forecast_future['Lower_90'].round(0).astype(int)
forecast_future['Upper_90'] = forecast_future['Upper_90'].round(0).astype(int)

print(f'30-day forecast from {forecast_future["Date"].min().date()} to {forecast_future["Date"].max().date()}')
display(forecast_future)

In [ ]:
# Visualize future forecast
fig, ax = plt.subplots(figsize=(14, 5))

# Show last 60 days of actual data for context
recent = df.tail(60)
ax.plot(recent['LogDate'], recent['Total_Daily_Keys'], 'o-', label='Actual (recent)', color='steelblue', markersize=3)

ax.plot(forecast_future['Date'], forecast_future['Forecast'], 's-', label='Forecast', color='darkorange', markersize=4)
ax.fill_between(
    forecast_future['Date'],
    forecast_future['Lower_90'],
    forecast_future['Upper_90'],
    alpha=0.2, color='darkorange', label='90% Confidence Interval'
)
ax.axvline(last_date, color='black', linestyle='--', alpha=0.5)

ax.set_title('30-Day DRM Key Demand Forecast')
ax.set_ylabel('DRM Keys')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Save model and forecast
OUTPUT_DIR = os.path.join('..', 'Dataset')

with open(os.path.join(OUTPUT_DIR, 'model_prophet_final.pkl'), 'wb') as f:
    pickle.dump(model_prophet_final, f)

forecast_future.to_csv(os.path.join(OUTPUT_DIR, 'forecast_30d.csv'), index=False)

# Save evaluation results
results.to_csv(os.path.join(OUTPUT_DIR, 'model_evaluation.csv'), index=False)

print('Saved:')
print(f'  → model_prophet_final.pkl')
print(f'  → forecast_30d.csv')
print(f'  → model_evaluation.csv')

In [ ]:
print('\n' + '=' * 55)
print('FORECASTING SUMMARY')
print('=' * 55)
print(f'Best model            : {best_model_name}')
print(f'Test MAPE             : {results.loc[results["MAPE (%)"].idxmin(), "MAPE (%)"]:.2f}%')
print(f'Test RMSE             : {results.loc[results["MAPE (%)"].idxmin(), "RMSE"]:.1f}')
print(f'Forecast period       : {forecast_future["Date"].min().date()} → {forecast_future["Date"].max().date()}')
print(f'Avg daily forecast    : {forecast_future["Forecast"].mean():,.0f} keys')
print(f'Total 30-day forecast : {forecast_future["Forecast"].sum():,} keys')
print(f'90% CI range (avg)    : [{forecast_future["Lower_90"].mean():,.0f}, {forecast_future["Upper_90"].mean():,.0f}]')
print('=' * 55)
print('\n→ Next: 04_recommendation_output.ipynb')